In [ ]:
#分红率
"""
隐含分红率（隐含基差率）计算 — L notebook (Wind API) 逻辑复现
用 akshare 替代 Wind
公式：q = r - ln(F/S) * 365 / 自然日数
"""

import akshare as ak
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ═══════════ 配置 ═══════════
INDEX_SYMBOL = "sh000852"       # 中证1000，sh000852；沪深300用sh000300
FUTURES_PREFIX = "IM"            # IM/IF/IC/IH
RISK_FREE_RATE = 0.014
VALUATION_DATE = "2026-08-24"
WINDOW_DAYS = 488
SIGMA_MULTIPLIER = 1.5

# ═══════════ 辅助函数 ═══════════

def get_third_friday(year, month):
    """中金所：合约月份第三个周五"""
    first_day = datetime(year, month, 1)
    days_to_friday = (4 - first_day.weekday()) % 7
    return first_day + timedelta(days=days_to_friday) + timedelta(weeks=2)


def get_window_contracts(prefix, valuation_date):
    """生成488交易日窗口内需要的季月合约
    488交易日 ≈ 730自然日，往前多拉一年确保覆盖早期合约"""
    val = valuation_date if isinstance(valuation_date, datetime) else datetime.strptime(valuation_date, "%Y-%m-%d")
    contracts = []
    for y in range(val.year - 3, val.year + 2):
        for m in [3, 6, 9, 12]:
            expiry = get_third_friday(y, m)
            days_from_val = (expiry - val).days
            # 往前覆盖~2.5年自然日（确保488交易日窗口内的旧合约不遗漏）
            if -910 < days_from_val < 730:
                code = f"{prefix}{y % 100:02d}{m:02d}"
                contracts.append(code)
    return sorted(set(contracts))


# ═══════════ 1. 用 akshare 读现货指数数据 ═══════════
print("=" * 60)
print(f"  读取指数数据: {INDEX_SYMBOL}")
# stock_zh_index_daily 返回 date, open, high, low, close, volume, ...
index_df = ak.stock_zh_index_daily(symbol=INDEX_SYMBOL)
index_df.rename(columns={"date": "date", "close": "close"}, inplace=True)
index_df["date"] = pd.to_datetime(index_df["date"])
index_df = index_df.sort_values("date").reset_index(drop=True)
# 只保留有期货数据的时段（2022-07 之后）
index_df = index_df[index_df["date"] >= "2022-01-01"].reset_index(drop=True)
print(f"  {len(index_df)} 行，{index_df['date'].iloc[0].date()} ~ {index_df['date'].iloc[-1].date()}")

# ═══════════ 2. 用 akshare 读期货数据 ═══════════
print(f"\n  读取期货合约: {FUTURES_PREFIX}")

# 只生成488天窗口内需要的合约
contracts_list = get_window_contracts(FUTURES_PREFIX, VALUATION_DATE)

all_futures = []
for code in contracts_list:
    try:
        df = ak.futures_zh_daily_sina(symbol=code)
        if df is None or len(df) == 0:
            continue
        df.rename(columns={"date": "date", "close": "futures_close"}, inplace=True)
        df["date"] = pd.to_datetime(df["date"])
        df["contract"] = code
        # 从合约代码推导到期日（第三个周五）
        y, m = 2000 + int(code[-4:-2]), int(code[-2:])
        df["expiry"] = get_third_friday(y, m)
        all_futures.append(df)
        print(f"    {code}: {len(df)} 行，到期 {df['expiry'].iloc[0].date()}")
    except Exception as e:
        # 已退市合约可能获取不到，跳过
        pass

if not all_futures:
    print("  ⚠ 没有获取到任何期货合约数据，请检查网络或 akshare 版本")
    exit()

futures_df = pd.concat(all_futures, ignore_index=True)
futures_df = futures_df.sort_values(["date", "contract"]).reset_index(drop=True)
print(f"  合计 {len(futures_df)} 行")

# ═══════════ 3. 逐日计算 q_benchmark ═══════════
print(f"\n  计算隐含分红率...")
fut_by_date = {d: grp for d, grp in futures_df.groupby("date")}

records = []
for row in index_df.itertuples():
    day = row.date
    S = row.close
    if pd.isna(S) or S <= 0:
        continue
    fut_today = fut_by_date.get(day, pd.DataFrame())
    if fut_today.empty:
        continue

    df = fut_today.copy()
    df["T_days"] = (df["expiry"] - day).dt.days
    df = df.sort_values("T_days")
    if df.empty:
        continue

    selected = df.tail(2)  # L notebook: df_code[-2:], 无 T>60 过滤
    qs = []
    for _, r2 in selected.iterrows():
        # L notebook: q = r - ln(F/S) * 365/ 自然日数
        q = RISK_FREE_RATE - np.log(r2["futures_close"] / S) * 365 / r2["T_days"]
        qs.append(q)

    if qs:
        records.append({"date": day, "q": float(np.mean(qs))})

q_df = pd.DataFrame(records).sort_values("date").reset_index(drop=True)
print(f"  有效数据：{len(q_df)} 条 ({q_df['date'].iloc[0].date()} ~ {q_df['date'].iloc[-1].date()})")

# ═══════════ 4. 取488天窗口 + 异常值剔除 ═══════════
val_date = pd.to_datetime(VALUATION_DATE)
q_window = q_df[q_df["date"] <= val_date].tail(WINDOW_DAYS).copy()
n_actual = len(q_window)
print(f"  使用 {n_actual} 个交易日数据")

values = q_window["q"].values
z = 1.96
mean_all = np.mean(values)
std_all = np.std(values, ddof=1)
filtered = [x for x in values if abs(x - mean_all) <= z * std_all]
n_removed = len(values) - len(filtered)

central = np.mean(filtered)
std_f = np.std(filtered, ddof=1)
upper = central + SIGMA_MULTIPLIER * std_f
lower = central - SIGMA_MULTIPLIER * std_f

# ═══════════ 5. 打印结果 ═══════════
print("\n" + "=" * 60)
print("  隐含分红率（隐含基差率）计算结果  [akshare 版]")
print("=" * 60)
print(f"  估值日：              {VALUATION_DATE}")
print(f"  品种：                {FUTURES_PREFIX}")
print(f"  无风险利率 r (FTP)：  {RISK_FREE_RATE*100:.2f}%")
print(f"  历史窗口：            {n_actual} 个交易日")
print(f"  剔除异常值（95%CI外）：{n_removed} 个，剩余 {len(filtered)} 个")
print("-" * 60)
print(f"  剔除前均值：          {mean_all*100:.2f}%")
print(f"  剔除前标准差：        {std_all*100:.4f}%")
print(f"  剔除后标准差：        {std_f*100:.4f}%")
print("-" * 60)
print(f"  >>> 隐含分红率中枢值：  {central*100:.2f}%")
print(f"  >>> 上边界 (+1.5σ)：   {upper*100:.2f}%")
print(f"  >>> 下边界 (-1.5σ)：   {lower*100:.2f}%")
print("=" * 60)

# # ═══════════ 6. 保存 CSV ═══════════
# summary = pd.DataFrame({
#     "指标": ["隐含分红率中枢值", "上边界(+1.5σ)", "下边界(-1.5σ)",
#              "剔除后标准差", "剔除前均值", "剔除前标准差",
#              "总天数", "剔除后天数"],
#     "值": [f"{central*100:.2f}%", f"{upper*100:.2f}%",
#            f"{lower*100:.2f}%", f"{std_f*100:.4f}%",
#            f"{mean_all*100:.2f}%", f"{std_all*100:.2f}%",
#            f"{len(values)}", f"{len(filtered)}"]
# })
# summary.to_csv("dividend_rate_summary_akshare.csv", index=False, encoding="utf-8-sig")
# print("\n已保存：dividend_rate_summary_akshare.csv")


  读取指数数据: sh000300
  1104 行，2022-01-04 ~ 2026-07-27

  读取期货合约: IF
    IF2403: 157 行，到期 2024-03-15
    IF2406: 162 行，到期 2024-06-21
    IF2409: 161 行，到期 2024-09-20
    IF2412: 164 行，到期 2024-12-20
    IF2503: 161 行，到期 2025-03-21
    IF2506: 163 行，到期 2025-06-20
    IF2509: 164 行，到期 2025-09-19
    IF2512: 165 行，到期 2025-12-19
    IF2603: 161 行，到期 2026-03-20
    IF2606: 163 行，到期 2026-06-19
    IF2609: 125 行，到期 2026-09-18
    IF2612: 67 行，到期 2026-12-18
    IF2703: 6 行，到期 2027-03-19
  合计 1819 行

  计算隐含分红率...
  有效数据：729 条 (2023-07-24 ~ 2026-07-27)
  使用 488 个交易日数据

  隐含分红率（隐含基差率）计算结果  [akshare 版]
  估值日：              2026-07-27
  品种：                IF
  无风险利率 r (FTP)：  2.00%
  历史窗口：            488 个交易日
  剔除异常值（95%CI外）：13 个，剩余 475 个
------------------------------------------------------------
  剔除前均值：          5.75%
  剔除前标准差：        3.0389%
  剔除后标准差：        2.6830%
------------------------------------------------------------
  >>> 隐含分红率中枢值：  5.70%
  >>> 上边界 (+1.5σ)：   9.72%
  >>> 下边界 (-1.5σ)：   1.6

In [1]:
import akshare as ak
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ═══════════ 配置 ═══════════
INDEX_SYMBOL = "sh000852"       # 中证1000，sh000852；沪深300用sh000300
FUTURES_PREFIX = "IM"            # IM/IF/IC/IH
RISK_FREE_RATE = 0.014
VALUATION_DATE = "2026-08-24"
WINDOW_DAYS = 488
SIGMA_MULTIPLIER = 1.5

# ═══════════ 辅助函数 ═══════════

def get_third_friday(year, month):
    """中金所：合约月份第三个周五"""
    first_day = datetime(year, month, 1)
    days_to_friday = (4 - first_day.weekday()) % 7
    return first_day + timedelta(days=days_to_friday) + timedelta(weeks=2)


def get_window_contracts(prefix, valuation_date):
    """生成488交易日窗口内需要的季月合约
    488交易日 ≈ 730自然日，往前多拉一年确保覆盖早期合约"""
    val = valuation_date if isinstance(valuation_date, datetime) else datetime.strptime(valuation_date, "%Y-%m-%d")
    contracts = []
    for y in range(val.year - 3, val.year + 2):
        for m in [3, 6, 9, 12]:
            expiry = get_third_friday(y, m)
            days_from_val = (expiry - val).days
            # 往前覆盖~2.5年自然日（确保488交易日窗口内的旧合约不遗漏）
            if -910 < days_from_val < 730:
                code = f"{prefix}{y % 100:02d}{m:02d}"
                contracts.append(code)
    return sorted(set(contracts))


# ═══════════ 1. 用 akshare 读现货指数数据 ═══════════
print("=" * 60)
print(f"  读取指数数据: {INDEX_SYMBOL}")
# stock_zh_index_daily 返回 date, open, high, low, close, volume, ...
index_df = ak.stock_zh_index_daily(symbol=INDEX_SYMBOL)
index_df.rename(columns={"date": "date", "close": "close"}, inplace=True)
index_df["date"] = pd.to_datetime(index_df["date"])
index_df = index_df.sort_values("date").reset_index(drop=True)
# 只保留有期货数据的时段（2022-07 之后）
index_df = index_df[index_df["date"] >= "2022-01-01"].reset_index(drop=True)
print(f"  {len(index_df)} 行，{index_df['date'].iloc[0].date()} ~ {index_df['date'].iloc[-1].date()}")

# ═══════════ 2. 用 akshare 读期货数据 ═══════════
print(f"\n  读取期货合约: {FUTURES_PREFIX}")

# 只生成488天窗口内需要的合约
contracts_list = get_window_contracts(FUTURES_PREFIX, VALUATION_DATE)

all_futures = []
for code in contracts_list:
    try:
        df = ak.futures_zh_daily_sina(symbol=code)
        if df is None or len(df) == 0:
            continue
        df.rename(columns={"date": "date", "close": "futures_close"}, inplace=True)
        df["date"] = pd.to_datetime(df["date"])
        df["contract"] = code
        # 从合约代码推导到期日（第三个周五）
        y, m = 2000 + int(code[-4:-2]), int(code[-2:])
        df["expiry"] = get_third_friday(y, m)
        all_futures.append(df)
        print(f"    {code}: {len(df)} 行，到期 {df['expiry'].iloc[0].date()}")
    except Exception as e:
        # 已退市合约可能获取不到，跳过
        pass

if not all_futures:
    print("  ⚠ 没有获取到任何期货合约数据，请检查网络或 akshare 版本")
    exit()

futures_df = pd.concat(all_futures, ignore_index=True)
futures_df = futures_df.sort_values(["date", "contract"]).reset_index(drop=True)
print(f"  合计 {len(futures_df)} 行")

  读取指数数据: sh000852
  1123 行，2022-01-04 ~ 2026-08-21

  读取期货合约: IM
    IM2403: 157 行，到期 2024-03-15
    IM2406: 162 行，到期 2024-06-21
    IM2409: 161 行，到期 2024-09-20
    IM2412: 164 行，到期 2024-12-20
    IM2503: 161 行，到期 2025-03-21
    IM2506: 163 行，到期 2025-06-20
    IM2509: 164 行，到期 2025-09-19
    IM2512: 165 行，到期 2025-12-19
    IM2603: 161 行，到期 2026-03-20
    IM2606: 163 行，到期 2026-06-19
    IM2609: 144 行，到期 2026-09-18
    IM2612: 86 行，到期 2026-12-18
    IM2703: 25 行，到期 2027-03-19
  合计 1876 行
